[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/01_eda_visualization/01_eda_visualization_solutions.ipynb)

# 01. 데이터 탐색과 시각화 — 연습 문제 해설

[01_eda_visualization.ipynb](01_eda_visualization.ipynb) 끝의 연습 문제 6개에 대한 정답 코드와
해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

본문과 같은 순서입니다. **문제 1~4는 1부(택시), 문제 5~6은 2부(타이타닉)** 범위입니다.

> **읽는 법** — 셀은 위에서부터 순서대로 실행해야 합니다(`Shift + Enter`). 실행 결과는 저장되어
> 있지 않으니 직접 실행해야 표와 그래프가 나타납니다. 맨 위의 **준비 셀들을 먼저 실행한 뒤**
> 원하는 문제로 건너뛰면 됩니다. 해설에 적힌 숫자는 실행하면 나오는 값입니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pandas seaborn matplotlib koreanize-matplotlib

### 준비 셀

아래 셀들은 본문과 같은 준비 코드입니다. **내용을 이해할 필요 없이 그대로 실행**하면 됩니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

본문 1부·2부에서 쓴 두 데이터를 한 번에 불러옵니다. **문제 1~4는 `trips`, 문제 5~6은 `titanic`** 을 씁니다.

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

titanic = sns.load_dataset("titanic")

print("trips  :", trips.shape)
print("titanic:", titanic.shape)

---

# 1부 — 택시 (`trips`)

## 문제 1. 하차 자치구 분포와 승차 자치구 비교

In [ ]:
sns.countplot(data=trips, x="dropoff_borough")
plt.title("하차 자치구")
plt.show()

그래프로는 정확한 비율을 알 수 없습니다. 승차와 하차를 **한 표에 나란히 놓고** 차이를 계산합니다.

In [ ]:
pickup_pct = (trips["pickup_borough"].value_counts(normalize=True) * 100).round(1)
dropoff_pct = (trips["dropoff_borough"].value_counts(normalize=True) * 100).round(1)

compare = pd.DataFrame({"승차(%)": pickup_pct, "하차(%)": dropoff_pct})
compare["차이"] = (compare["하차(%)"] - compare["승차(%)"]).round(1)
compare

**해설**

| 자치구 | 승차 | 하차 | 차이 |
|---|---|---|---|
| Manhattan | 82.2% | 81.5% | -0.7 |
| Queens | 10.3% | 8.5% | -1.8 |
| Brooklyn | 6.0% | 7.8% | **+1.8** |
| Bronx | 1.5% | 2.1% | +0.6 |
| Staten Island | — | 0.03% (2건) | **승차에는 아예 없음** |

- **Staten Island가 하차 목록에만 있습니다.** 승차 데이터에는 한 건도 없는데 하차는 2건입니다.
  섬이라 택시 진입은 있어도 출발은 거의 없다는 뜻입니다. 이렇게 **한쪽에만 존재하는 범주**는
  나중에 원-핫 인코딩을 할 때 문제가 됩니다. 학습 데이터에는 없고 검증 데이터에만 있는 범주가
  생기면 컬럼 구성이 어긋나기 때문입니다. (02번에서 다룹니다.)
- Brooklyn은 승차보다 하차가 1.8%p 많습니다. 맨해튼에서 브루클린으로 나가는 이동이
  그 반대보다 많다는 뜻입니다.
- 승차·하차 모두 맨해튼이 80% 이상이라, 이 두 컬럼은 **정보량이 크지 않습니다.**
  03번의 변수중요도에서 실제로 낮게 나오는지 확인해보면 좋습니다.

## 문제 2. 결제 수단별 팁 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(data=trips, x="payment", y="tip", ax=axes[0])
axes[0].set_title("결제 수단별 팁 (boxplot)")

sns.histplot(data=trips, x="tip", hue="payment", bins=40, ax=axes[1])
axes[1].set_title("결제 수단별 팁 분포 (histplot)")

plt.tight_layout()
plt.show()

그래프에서 본 차이를 숫자로 확인합니다.

In [ ]:
trips.groupby("payment")["tip"].agg(["mean", "median", "count"]).round(3)

현금 쪽은 평균도 중앙값도 0입니다. **정말 한 건도 없는지** 0인 비율을 세어봅시다.

In [ ]:
# 팁이 정확히 0인 비율
(trips.groupby("payment")["tip"].apply(lambda s: (s == 0).mean()) * 100).round(1)

**해설**

| 결제 수단 | 평균 | 중앙값 | 팁이 0인 비율 |
|---|---|---|---|
| cash | **0.000** | **0.0** | **100.0%** |
| credit card | 2.782 | 2.2 | 9.9% |

**현금 결제는 팁이 단 한 건도 기록되어 있지 않습니다.** 평균도 중앙값도 정확히 0입니다.

여기서 "현금으로 내는 사람은 팁을 안 준다"고 결론 내리면 틀립니다. 진짜 이유는
**현금 팁은 승객이 기사에게 직접 건네므로 미터기에 입력되지 않는다**는 것입니다.
즉 이 0은 "팁이 없었다"가 아니라 **"기록되지 않았다"** 는 뜻이고, 사실상 결측치에 가깝습니다.

이런 것을 **측정 방식이 만들어낸 패턴(measurement artifact)** 이라고 합니다. 데이터가 어떻게
수집되었는지 모르면 완전히 잘못된 해석을 하게 되는 대표적인 예입니다.

실무적으로도 중요합니다. 만약 `tip`으로 `total`을 예측하는 모델을 만든다면, 모델은
"현금이면 팁 0"이라는 규칙을 학습해서 겉으로는 정확도가 높게 나옵니다. 하지만 이건 승객의
행동을 배운 게 아니라 **결제 시스템의 기록 방식을 배운 것**이라, 현금 팁을 기록하는 시스템으로
바뀌는 순간 모델은 무너집니다.

## 문제 3. 잘못된 코드 고치기

**문제로 주어진 코드**

```python
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 4))
sns.countplot(data=trips, x="pickup_borough", ax=axes[1])
sns.jointplot(data=trips, x="distance", y="duration", ax=axes[2])
axes[0].title("승차 자치구")
plt.show()
```

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 4))

sns.countplot(data=trips, x="pickup_borough", ax=axes[0])
sns.scatterplot(data=trips, x="distance", y="duration", alpha=0.3, ax=axes[1])

axes[0].set_title("승차 자치구")
axes[1].set_title("거리 - 시간")

plt.tight_layout()
plt.show()

**해설 — 잘못된 곳 세 군데**

| # | 잘못된 코드 | 고친 코드 | 이유 |
|---|---|---|---|
| 1 | `ax=axes[1]` (countplot) | `ax=axes[0]` | 제목을 `axes[0]`에 달려고 했으니 첫 칸에 그리는 게 의도. 에러는 안 나지만 제목이 빈 칸에 붙습니다 |
| 2 | `sns.jointplot(..., ax=axes[2])` | `sns.scatterplot(..., ax=axes[1])` | `jointplot`은 **figure-level** 함수라 `ax=`를 받지 않습니다. 또 `ncols=2`이므로 인덱스는 0과 1뿐이라 `axes[2]`는 범위를 벗어납니다 |
| 3 | `axes[0].title(...)` | `axes[0].set_title(...)` | `Axes` 객체에서 `title`은 **속성**이지 함수가 아닙니다. 제목을 다는 함수는 `set_title` |

에러 메시지는 이런 순서로 나타납니다.

1. `IndexError: index 2 is out of bounds for axis 0 with size 2` → 인덱스 범위
2. `TypeError: jointplot() got an unexpected keyword argument 'ax'` → 함수 선택
3. `TypeError: 'Text' object is not callable` → `title` vs `set_title`

**에러가 사라졌다고 끝이 아닙니다.** 1번은 에러를 내지 않기 때문에, 그림을 보고
"의도한 대로 나왔는가"까지 확인해야 발견됩니다.

## 문제 4. `passengers`가 0인 96건, 제거해야 할까?

정답이 하나로 정해진 문제는 아닙니다. **근거를 세워 판단하는 연습**이 목적입니다.

In [ ]:
zero = trips[trips["passengers"] == 0]
rest = trips[trips["passengers"] > 0]

print(f"승객 0명: {len(zero)}건 ({len(zero) / len(trips) * 100:.1f}%)")
print()
print("=== 승객 0명 ===")
print(zero[["distance", "duration", "fare", "total"]].describe().round(2))
print()
print("=== 나머지 ===")
print(rest[["distance", "duration", "fare", "total"]].describe().round(2))

세 컬럼의 분포를 **승객 0명 그룹과 나머지로 나눠** 나란히 그립니다. 두 상자가 비슷하다면
"이 행들은 정상 운행"이라는 근거가 됩니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

flag = trips["passengers"] == 0
labeled = trips.assign(승객0명=flag.map({True: "0명", False: "1명 이상"}))

for ax, col in zip(axes, ["distance", "duration", "fare"]):
    sns.boxplot(data=labeled, x="승객0명", y=col, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

거리·시간·요금 외의 컬럼에서도 특이한 점이 있는지 확인합니다.

In [ ]:
# 다른 컬럼에서도 특이한 점이 있는지 확인
print(zero["color"].value_counts())
print()
print(zero["payment"].value_counts(dropna=False))

**해설 — 판단의 근거**

먼저 **"제거하자"는 쪽 근거**입니다.

- 승객 0명인 택시 운행은 물리적으로 말이 안 됩니다. 명백히 잘못된 값입니다.
- 96건은 전체의 1.5%라 지워도 데이터 손실이 크지 않습니다.

하지만 **실제로 비교해보면 "남기자"는 쪽이 더 설득력 있습니다.**

| 지표 | 승객 0명 | 나머지 |
|---|---|---|
| `distance` 중앙값 | 1.60 | 1.64 |
| `duration` 중앙값 | 11.52 | 10.90 |
| `fare` 중앙값 | 9.25 | 9.50 |

**거리·시간·요금 분포가 나머지와 거의 똑같습니다.** 이건 중요한 신호입니다.

- 만약 이 행들이 "실제로 일어나지 않은 운행"이라면 거리나 요금이 0에 몰려 있어야 합니다.
  그런데 정상 운행과 구분이 안 됩니다.
- 즉 **운행 자체는 정상적으로 일어났고, `passengers` 컬럼만 기록되지 않은 것**으로 보입니다.
  미터기에 승객 수를 입력하지 않은 채 운행한 경우겠죠.
- 그렇다면 이건 "이상치"가 아니라 **`passengers` 한 컬럼에만 있는 결측치**입니다.
  나머지 컬럼(`distance`, `duration`, `fare`)의 정보는 멀쩡하므로, 행 전체를 버리는 건 손해입니다.

**결론과 대안**

우리의 목표는 `duration`(이동 시간) 예측입니다. `passengers`는 이동 시간에 거의 영향을 주지 않는
변수이므로, 이 96건 때문에 고민할 필요 자체가 크지 않습니다. 선택지는 이렇습니다.

1. **그대로 둔다** — `passengers`의 영향이 미미하니 0을 하나의 값으로 취급 (가장 간단, 이 시리즈의 선택)
2. **`passengers`만 결측치로 바꿔 중앙값으로 채운다** — `df.loc[df["passengers"] == 0, "passengers"] = np.nan` 후 대체
3. **행을 제거한다** — `passengers`가 핵심 변수인 다른 문제(예: 승객 수별 요금 정책 분석)라면 타당

**핵심은 "이상해 보이니까 지운다"가 아니라, 그 행들이 다른 컬럼에서 어떻게 생겼는지 확인한 뒤
목표에 비추어 판단한다**는 것입니다. 본문에서 제거한 `speed >= 60`인 행은 9.4마일을 7초에 갔다는
**물리적으로 불가능한** 기록이라 성격이 다릅니다.

---

# 2부 — 타이타닉 (`titanic`)

## 문제 5. 탑승 항구 분포와 등급별 요금을 나란히

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 5))

sns.countplot(data=titanic, x="embark_town", ax=axes[0])
axes[0].set_title("탑승 항구")
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")

sns.boxplot(data=titanic, x="pclass", y="fare", ax=axes[1])
axes[1].set_title("등급별 요금")

plt.tight_layout()
plt.show()

**해설**

체크할 점 네 가지입니다.

1. **가로 배치는 `nrows=1, ncols=2`.** 세로로 쌓는 `nrows=2, ncols=1`과 헷갈리기 쉽습니다.
2. **각 그래프에 `ax=axes[i]`를 지정해야 합니다.** 빼먹으면 두 그래프가 첫 칸에 겹쳐 그려지고
   두 번째 칸은 빈 채로 남습니다.
3. **제목은 `axes[i].set_title()`.** `plt.title()`은 마지막에 활성화된 축에만 붙어서,
   두 그래프 모두에 제목을 달려고 하면 뒤엣것만 반영됩니다.
4. **라벨 기울이기는 `plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")`.**
   `ha="right"`(수평 정렬)까지 넣어야 기울인 글자의 끝이 눈금에 맞습니다.

내용 면에서는 1등급 요금이 압도적으로 높고 위쪽 이상치도 1등급에 몰려 있습니다.
`fare`의 이상치가 사실은 "오류"가 아니라 **1등급 특실 요금이라는 실제 값**이라는 뜻입니다.
이런 경우 무작정 제거하는 게 옳은지는 판단이 필요합니다 — 02번에서 다시 이야기합니다.

## 문제 6. 타이타닉 상관계수 히트맵

In [ ]:
corr = titanic.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("타이타닉 상관계수")
plt.show()

히트맵에서 `survived` 줄만 뽑아 **절댓값 순으로** 정렬하면 순위가 한눈에 보입니다.

In [ ]:
corr["survived"].drop("survived").sort_values(key=abs, ascending=False)

**해설**

| 컬럼 | 상관계수 |
|---|---|
| `adult_male` | **-0.557** |
| `pclass` | -0.339 |
| `fare` | +0.257 |
| `alone` | -0.203 |
| `parch` | +0.082 |
| `age` | -0.077 |
| `sibsp` | -0.035 |

**절댓값이 가장 큰 것은 `adult_male` (-0.557)** 입니다. 성인 남성일수록 생존 확률이 낮았다는 뜻이고,
본문 2부에서 본 **여성 74.2% vs 남성 18.9%** 라는 생존율 차이가 상관계수로 표현된 것입니다.
`pclass`(-0.339) 역시 **1등급 63.0% vs 3등급 24.2%** 와 대응됩니다.

주의할 점 세 가지입니다.

- **`pclass`의 부호가 음수인 이유**: 1등급이 숫자 `1`, 3등급이 숫자 `3`이라 "숫자가 작을수록 좋은 등급"입니다.
  그래서 등급이 좋을수록 생존율이 높다는 관계가 **음의 상관**으로 나타납니다. 숫자로 저장된 범주형을
  다룰 때 늘 확인해야 하는 부분입니다.
- **bool 컬럼도 계산에 들어갑니다.** pandas는 `adult_male`, `alone` 같은 bool을 True=1, False=0으로
  보고 수치형으로 취급합니다. 반면 문자열인 `sex`, `who`는 히트맵에서 아예 빠집니다.
  **가장 중요한 변수인 `sex`가 표에 안 보인다고 해서 중요하지 않은 게 아닙니다** —
  단지 상관계수로 표현할 수 없는 자료형일 뿐입니다.
- **`numeric_only=True`가 없으면 에러**가 납니다. 문자열끼리는 상관계수를 계산할 수 없기 때문입니다.

여기서 나온 순위를 [03_tree_models](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/03_tree_models/03_tree_models.ipynb)의 **변수중요도**와
비교해보면 흥미롭습니다. 상관계수는 "타깃과의 1:1 직선 관계"만 보지만, 변수중요도는 다른 변수와의
조합 효과까지 반영하므로 순위가 달라질 수 있습니다.

---

다음 노트북: [02_preprocessing.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/02_preprocessing/02_preprocessing.ipynb)